<div align="center" style="background-color: #FFF3E0; padding: 20px; border-radius: 12px; border-right: 5px solid #E67E22; border-left: 5px solid #E67E22; margin-bottom: 20px;">
  <h1 style="color: #D35400; margin: 0; font-size: 28px;">
    🏢 پروژه فناوری اطلاعات - فاز ۱: پیش‌پردازش داده‌ها
  </h1>
  <p style="color: #6E2C00; font-size: 18px; margin: 10px 0 0 0;">
    👥 اعضای گروه: مهدی اباذری اقدم | الشن کمانی معروفی | سیدامیرحسین محمدی
  </p>
</div>

<div align="center" style="background-color: #E8F8F5; padding: 15px; border-radius: 12px; border-right: 5px solid #1ABC9C; border-left: 5px solid #1ABC9C;">
  <h2 style="color: #0E6655; margin: 0;">
    📦 بخش ۰: فراخوانی کتابخانه‌ها و داده‌ها
  </h2>
</div>

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import warnings
import os
warnings.filterwarnings('ignore')
from pathlib import Path
DATA_DIR = Path("dirty_data")
if not DATA_DIR.is_dir():
    DATA_DIR = Path("Phase_1_DataPreprocessing_python") / "dirty_data"


In [ ]:
if not os.path.exists('data_clean'):
    os.makedirs('data_clean')

file_names = ['Orders', 'Order_Details', 'Customers', 'Employees', 'Products', 'Categories', 
              'Suppliers', 'Shippers', 'Region', 'Territories', 'EmployeeTerritories', 
              'CustomerDemographics', 'CustomerCustomerDemo']

datasets = {name: pd.read_csv(DATA_DIR / f"{name}.csv") for name in file_names}
report_log = []

def log(msg):
    print(msg)
    report_log.append(msg)

print("--- Data loaded into 'datasets' dictionary ---")



<div align="center" style="background-color: #E8F8F5; padding: 15px; border-radius: 12px; border-right: 5px solid #1ABC9C; border-left: 5px solid #1ABC9C;">
  <h2 style="color: #0E6655; margin: 0;">
 🧹 بخش 1 (2.3): پاکسازی داده‌ها و جایگزینی مقادیر گسسته
  </h2>
</div>

In [ ]:
log("--- 1. Calculating missing value percentages ---")
missing_info = {}

for name, df in datasets.items():
    if len(df) == 0:
        log(f"Warning: Table '{name}' is completely empty (0 rows).")
        continue
        
    missing_percent = (df.isnull().sum() / len(df)) * 100
    # فیلتر کردن ستون‌هایی که حداقل یک داده گمشده دارند
    missing_cols = missing_percent[missing_percent > 0]
    
    if not missing_cols.empty:
        log(f"\n* Table '{name}':")
        for col, pct in missing_cols.items():
            log(f"  - Column '{col}': {pct:.2f}% missing")
        missing_info[name] = missing_cols

if not missing_info:
    log("No missing values were found in the tables!")


In [ ]:
log("--- 2. Checking and handling columns with more than 30% missing data ---")

for name in list(datasets.keys()):
    df = datasets[name]
    
    # جلوگیری از خطای جداول خالی
    if len(df) == 0:
        continue
        
    pct = (df.isnull().sum() / len(df)) * 100
    high_missing_cols = pct[pct > 30].index.tolist()
    
    if high_missing_cols:
        log(f"\n* Table '{name}': Handling columns with >30% missing: {high_missing_cols}")
        for col in high_missing_cols:
            
            # تصمیم‌گیری بر اساس منطق کسب‌وکار
            if col in ['ShipRegion', 'Region']:
                # دلیل: بسیاری از کشورها ایالت ندارند. حفظ این داده برای کشورهای ایالت‌دار مهم است.
                datasets[name][col] = df[col].fillna('Unknown')
                log(f"  - '{col}': Imputed with 'Unknown'")
                
            elif col == 'Fax':
                # دلیل: نداشتن فکس به معنای خرابی داده نیست.
                datasets[name][col] = df[col].fillna('No Fax')
                log(f"  - '{col}': Imputed with 'No Fax'")
                
            elif col == 'QuantityPerUnit':
                # ستون مهم در محصولات
                datasets[name][col] = df[col].fillna('Not Specified')
                log(f"  - '{col}': Imputed with 'Not Specified'")
                
            elif col == 'Description':
                # توضیحات دسته‌بندی
                datasets[name][col] = df[col].fillna('No Description')
                log(f"  - '{col}': Imputed with 'No Description'")
                
            else:
                # اگر ستونی بود که توجیهی برای نگهداری نداشت، حذف می‌شود
                datasets[name] = datasets[name].drop(columns=[col])
                log(f"  - '{col}': Dropped (No business logic to retain)")
                
        # بروزرسانی دیکشنری missing_info برای ستون‌هایی که پردازش شدند
        if name in missing_info:
            missing_info[name] = missing_info[name].drop(index=high_missing_cols, errors='ignore')
            
    else:
        log(f"+ Table '{name}': No columns exceeded 30% missing threshold.")

log("--- Data quality check (>30% rule) complete. ---")


In [ ]:
log("--- 3. Comparing Mean vs Median for Numeric Imputation ---")

for name in list(datasets.keys()):
    df = datasets[name]
    
    # جلوگیری از بررسی جداول خالی
    if len(df) == 0:
        continue
        
    num_cols = df.select_dtypes(include=[np.number]).columns
    missing_num_cols = [col for col in num_cols if df[col].isnull().any()]
    
    if missing_num_cols:
        log(f"\n* Table '{name}':")
        
        for col in missing_num_cols:
            # ۱. جلوگیری از میانگین‌گیری برای ستون‌های کلید و شناسه (ID / Foreign Keys)
            if col.lower().endswith('id') or col.lower() in ['shipvia', 'postalcode','reportsto']:
                mode_val = df[col].mode()[0] if not df[col].mode().empty else 0
                datasets[name][col] = df[col].fillna(mode_val)
                log(f"  [!] Col '{col}' is an ID/Key. Imputed with Mode ({mode_val}) instead of Mean/Median.")
                continue
            
            # ۲. محاسبه شاخص‌های آماری
            mean_val = df[col].mean()
            median_val = df[col].median()
            skewness = df[col].skew() # محاسبه چولگی برای تشخیص داده‌های پرت
            
            # ۳. تصمیم‌گیری خودکار بر اساس چولگی (Skewness)
            # اگر چولگی بزرگتر از ۱ یا کوچکتر از ۱- باشد، یعنی داده‌ها پرت دارند و میانه بهتر است
            if abs(skewness) > 1.0:
                chosen_method = "Median"
                fill_val = median_val
                reason = f"High skewness ({skewness:.2f})"
            else:
                chosen_method = "Mean"
                fill_val = mean_val
                reason = f"Symmetric dist (skew: {skewness:.2f})"
                
            # ۴. جایگزینی مقادیر
            datasets[name][col] = df[col].fillna(fill_val)
            
            # ۵. ثبت گزارش ساختارمند و زیبا
            log(f"  - Col: '{col:<15}' | Mean: {mean_val:<8.2f} | Median: {median_val:<8.2f} -> Applied: {chosen_method:<6} ({reason})")
            
    else:
        log(f"+ Table '{name}': No missing numeric values.")

log("\n--- Numeric missing values handled successfully. ---")


In [ ]:
#حذف ردیف‌های کاملا تکراری
print("--- 4. Checking and removing duplicate records ---")
for name in list(datasets.keys()):
    initial_rows = datasets[name].shape[0]
    datasets[name] = datasets[name].drop_duplicates()
    final_rows = datasets[name].shape[0]
    
    deleted = initial_rows - final_rows
    if deleted > 0:
        log(f"- Table '{name}': {deleted} duplicate records removed.")
    else:
        log(f"+ Table '{name}': No duplicates found.")


print()
print("\n Duplicate record cleanup for all 13 tables has been completed.")


In [ ]:
import numpy as np

print("--- 5. Cleaning Text Columns ---")

skip_keywords = ['ID', 'Id', 'Phone', 'Fax', 'Postal', 'Code', 'Zip']

for name in list(datasets.keys()):
    df = datasets[name].copy() 
    text_cols = df.select_dtypes(include=['object']).columns
    
    for col in text_cols:
        # حذف فاصله‌ها و کاراکترهای اضافی
        df[col] = df[col].astype(str).str.replace(r'[\t\n\r]', ' ', regex=True)
        df[col] = df[col].str.replace(r'\s+', ' ', regex=True).str.strip()
        
        # یکسان‌سازی فرمت حروف
        if not any(kw in col for kw in skip_keywords):
            df[col] = df[col].str.title() 
            
        # تبدیل رشته‌های خالی کاذب به NaN (حذف 'Unknown' از این لیست)
        # دقت کنید که واژه‌هایی مثل 'No Fax' یا 'Not Specified' که در فاز قبل ساختید هم نباید اینجا باشند
        df[col] = df[col].replace(
            ['nan', 'None', 'Null', ''], np.nan
        )
        
    datasets[name] = df 
    
print("--- Text columns successfully cleaned in all tables. ---\n")


In [ ]:
#3.2 حذف یا اصلاح داده منفی یا صفر در قیمت و تعداد

print("--- 6. Checking and Fixing Zero/Negative Values in Order_Details ---")

# کپی گرفتن از داده‌های تمیز شده در مرحله قبل جهت پردازش
od = datasets['Order_Details'].copy()
# حذف ردیف‌های تکراری از محصولات برای ساخت دیکشنری قیمت (جلوگیری از خطای نگاشت)
products = datasets['Products'].drop_duplicates(subset=['ProductID'])
product_prices = products.set_index('ProductID')['UnitPrice']

# شناسایی ماسک‌های خطا (قیمت یا تعداد صفر و منفی)
invalid_price_mask = od['UnitPrice'] <= 0
invalid_qty_mask = od['Quantity'] <= 0

# 1. استراتژی اصلاح قیمت: جایگزینی قیمت‌های نامعتبر با قیمت اصلی از جدول محصولات
if invalid_price_mask.any():
    invalid_count = int(invalid_price_mask.sum())
    log(f"- Table 'Order_Details': {invalid_count} records with zero/negative UnitPrice. Fixing via product catalog.")
    # نگاشت قیمت صحیح بر اساس ProductID
    od.loc[invalid_price_mask, 'UnitPrice'] = od.loc[invalid_price_mask, 'ProductID'].map(product_prices)
else:
    log("+ Table 'Order_Details': UnitPrice column is clean.")

# 2. استراتژی اصلاح تعداد: تبدیل مقادیر منفی به قدر مطلق و حذف ردیف‌های صفر
if invalid_qty_mask.any():
    # اصلاح مقادیر منفی (تبدیل به مثبت)
    neg_mask = od['Quantity'] < 0
    neg_count = int(neg_mask.sum())
    if neg_count > 0:
        od.loc[neg_mask, 'Quantity'] = od.loc[neg_mask, 'Quantity'].abs()
        log(f"- Table 'Order_Details': {neg_count} negative Quantity values converted to absolute.")
    
    # حذف ردیف‌هایی که تعداد آن‌ها صفر است (غیرمنطقی برای سفارش)
    zero_mask = od['Quantity'] == 0
    zero_count = int(zero_mask.sum())
    if zero_count > 0:
        od = od[~zero_mask]
        log(f"- Table 'Order_Details': {zero_count} records with Quantity=0 dropped.")
else:
    log("+ Table 'Order_Details': Quantity column is clean.")

# ذخیره نهایی تغییرات در دیکشنری اصلی
datasets['Order_Details'] = od
print("--- Order_Details validation complete. ---")
print()
print("--- End of Section 3.2 ---")


<div align="center" style="background-color: #E8F8F5; padding: 15px; border-radius: 12px; border-right: 5px solid #1ABC9C; border-left: 5px solid #1ABC9C;">
  <h2 style="color: #0E6655; margin: 0;">
    🔧 بخش 2 (3.3): اصلاح نوع داده‌ها
  </h2>
</div>

In [ ]:
#3.3 تبدیل ستون های تاریخ به نوع داده تاریخ و زمان

print("--- 1. Converting Date Columns to Datetime Type ---")
print()
# مشخص کردن ستون‌های تاریخ در جداول مختلف
date_columns = {
    'Orders': ['OrderDate', 'RequiredDate', 'ShippedDate'],
    'Employees': ['BirthDate', 'HireDate']
}

for table_name, cols in date_columns.items():
    if table_name in datasets:
        df = datasets[table_name]
        for col in cols:
            if col in df.columns:
                # تبدیل مقادیر به فرمت دیت تایم پایتون
                df[col] = pd.to_datetime(df[col], errors='coerce')
                print(f"* Column '{col}' in '{table_name}' converted to Datetime.")
print()
print("--- Date conversion complete. ---")



In [ ]:
#3.3 شناسایی ستون‌های مشکوک به عدد خوانده شدن تلفن کد پستی و ...
print("--- 2.1.  Identifying Misinterpreted Columns ---")
print()
suspicious_cols = {}

for name, df in datasets.items():
    # ستون‌هایی که باید متنی باشند (مثل تلفن، کدپستی، IDها)
    target_cols = ['Phone', 'Fax', 'Postal', 'Code', 'Zip', 'ID']
    
    for col in df.columns:
        # اگر ستون شامل کلمات کلیدی است ولی در حال حاضر عددی است (int/float)
        if any(kw in col for kw in target_cols) and pd.api.types.is_numeric_dtype(df[col]):
            if name not in suspicious_cols: suspicious_cols[name] = []
            suspicious_cols[name].append(col)
            print(f"* Column '{col}' in table '{name}' is currently numeric but should be text.")

if not suspicious_cols:
    print("- No misidentified columns found.")



In [ ]:
#3.3 تبدیل ستون های مشکوک(خوانده شده بصورت عدد) به رشته
print("\n--- 2.2. Converting Misidentified Columns to String ---")
print()

for name, cols in suspicious_cols.items():
    # دریافت کپی از دیتافریم
    df = datasets[name]
    for col in cols:
        # تبدیل به رشته و حذف پسوند اعشاری .0
        df[col] = df[col].astype(str).str.replace(r'\.0$', '', regex=True)
        # جایگزینی مقادیر خالی به درستی
        df[col] = df[col].replace({'nan': np.nan, 'None': np.nan})
        print(f"* Column '{col}' in '{name}' successfully converted to String.")
    
    # ذخیره مجدد دیتافریم اصلاح شده در دیکشنری اصلی
    datasets[name] = df

print()
print("--- Data type conversion complete. ---")


In [ ]:
#3.3 شناسایی ستون‌های متنی که باید عددی باشند

print("--- 3.1. Identifying Misinterpreted Text Columns (Should be Numeric) ---")

numeric_misidentified = {}

for name, df in datasets.items():
    for col in df.columns:
        # اگر ستون متنی است
        if df[col].dtype == 'object':
            # بررسی اینکه آیا مقادیر آن شامل ارقام هستند (بعد از حذف کاراکترهای اضافه)
            # ما سعی می‌کنیم این ستون‌ها را به عدد تبدیل کنیم، اگر موفق شد، یعنی مشکوک است
            clean_series = df[col].astype(str).str.replace(r'[$,]', '', regex=True)
            if pd.to_numeric(clean_series, errors='coerce').notnull().mean() > 0.8:
                if name not in numeric_misidentified: numeric_misidentified[name] = []
                numeric_misidentified[name].append(col)
                print(f"- Column '{col}' in table '{name}' is text but looks numeric!")

if not numeric_misidentified:
    print(" No misidentified numeric columns found.")




In [ ]:
#3.3 اصلاح و تبدیل به نوع عددی نامریک
print("\n--- 3.2.: Converting Misidentified Text Columns to Numeric ---")
print()
for name, cols in numeric_misidentified.items():
    df = datasets[name]
    for col in cols:
        #حذف کاراکترهای غیرعددی مثل $  
        df[col] = df[col].astype(str).str.replace(r'[$,]', '', regex=True)
        #  تبدیل به عددی واقعی
        df[col] = pd.to_numeric(df[col], errors='coerce')
        print(f"* Column '{col}' in '{name}' successfully converted to Numeric.")
print()
print("--- Data type conversion complete. ---")





In [ ]:
#3.3 بررسی وجود مقادیر غیر عددی یا تهی در ستون های شناس
print("--- 4.1. Identifying ID Issues (Identification Only) ---")

for name in list(datasets.keys()):
    df = datasets[name]
    id_cols = [col for col in df.columns if 'ID' in col]
    
    for col in id_cols:
        null_count = df[col].isnull().sum()
        
        # بررسی مقادیر غیر عددی فقط برای مقادیر موجود
        non_null_series = df[col].dropna().astype(str)
        non_numeric = non_null_series.str.contains(r'[^0-9.]', regex=True).sum()
        
        if null_count > 0 or non_numeric > 0:
            log(f"- Table '{name}': Issues in ID column '{col}' -> Nulls={null_count}, Non-numeric={non_numeric}")
        else:
            log(f"+ Table '{name}': ID column '{col}' is clean.")

print("--- ID identification complete. ---")


In [ ]:
print("\n--- 4.2. Cleaning ID Columns (Decision Executed) ---")
print()

# لیست ستون‌های شناسه‌ای که ذاتاً متنی هستند و نباید به عدد تبدیل شوند
string_id_cols = ['CustomerID', 'CustomerTypeID']

for name in list(datasets.keys()):
    df = datasets[name]
    
    # پیدا کردن ستون‌های دارای ID به جز آن‌هایی که متنی هستند
    id_cols = [col for col in df.columns if 'ID' in col and col not in string_id_cols]
    
    for col in id_cols:
        initial_count = len(df)
        df[col] = pd.to_numeric(df[col], errors='coerce')
        df = df.dropna(subset=[col])
        df[col] = df[col].astype('Int64')
        
        deleted = initial_count - len(df)
        if deleted > 0:
            log(f"- Table '{name}': Column '{col}' cleaned. {deleted} rows with invalid/null ID removed.")
        else:
            log(f"+ Table '{name}': Column '{col}' validated as Int64.")
    
    datasets[name] = df
print("--- ID column cleanup complete. ---")

print()
print("--- Section 3.4 Completed Successfully ---")


<div align="center" style="background-color: #E8F8F5; padding: 15px; border-radius: 12px; border-right: 5px solid #1ABC9C; border-left: 5px solid #1ABC9C;">
  <h2 style="color: #0E6655; margin: 0;">
    ⚙️ بخش 3 (4.3): تبدیل و مهندسی ویژگی‌ها 
  </h2>
</div>

In [ ]:
# 3.4: مقیاس‌بندی متغیرهای عددی و مهندسی ویژگی

print("--- 1. Scaling Numeric Variables (Min-Max) ---")
print()

# تابع کمکی برای مقیاس‌بندی Min-Max جهت جلوگیری از تکرار کد و خطای تقسیم بر صفر
def min_max_scale(series):
    col_min = series.min()
    col_max = series.max()
    if col_max == col_min:
        return 0.0 # یا می‌تواند همان مقادیر اولیه برگردانده شود
    return (series - col_min) / (col_max - col_min)

# اعمال تغییرات روی جدول سفارشات (Orders)
if 'Orders' in datasets:
    df = datasets['Orders']
    # ایجاد ستون جدید به جای جایگزینی (حفظ دیتای اصلی برای SQL)
    df['Freight_Scaled'] = min_max_scale(df['Freight'])
    datasets['Orders'] = df

# اعمال تغییرات روی جدول جزئیات سفارش (Order_Details)
if 'Order_Details' in datasets:
    df = datasets['Order_Details']
    df['UnitPrice_Scaled'] = min_max_scale(df['UnitPrice'])
    df['Quantity_Scaled'] = min_max_scale(df['Quantity'])
    # ستون Discount معمولاً خودش یک نرخ بین 0 و 1 است، اما در صورت نیاز اسکیل می‌شود
    df['Discount_Scaled'] = min_max_scale(df['Discount'])
    datasets['Order_Details'] = df

# اعمال تغییرات روی جدول محصولات (Products)
if 'Products' in datasets:
    df = datasets['Products']
    df['UnitPrice_Scaled'] = min_max_scale(df['UnitPrice'])
    df['UnitsInStock_Scaled'] = min_max_scale(df['UnitsInStock'])
    df['UnitsOnOrder_Scaled'] = min_max_scale(df['UnitsOnOrder'])
    datasets['Products'] = df

print("--- Numeric variables scaling complete. ---")


In [ ]:
print("--- 2. Extracting Date Features (Year, Month, Day, DayOfWeek) ---")

if 'Orders' in datasets:
    df = datasets['Orders']
    
    # تبدیل ستون تاریخ به نوع datetime 
    df['OrderDate'] = pd.to_datetime(df['OrderDate'], errors='coerce')
    
    # استخراج سال، ماه و روز و تبدیل به Int64 (جلوگیری از تبدیل شدن به اعشاری در صورت وجود مقادیر خالی)
    df['OrderYear'] = df['OrderDate'].dt.year.astype('Int64')
    df['OrderMonth'] = df['OrderDate'].dt.month.astype('Int64')
    df['OrderDay'] = df['OrderDate'].dt.day.astype('Int64')
    
    # استخراج روز هفته (تنظیم شنبه = 0 تا جمعه = 6)
    df['OrderDayOfWeek'] = ((df['OrderDate'].dt.dayofweek + 2) % 7).astype('Int64')
    
    datasets['Orders'] = df

print("--- Date features extraction complete. ---")


In [ ]:
print("--- 3. Creating OrderValue Feature ---")

if 'Order_Details' in datasets:
    df_od = datasets['Order_Details']
    
    # تبدیل به نوع عددی 
    df_od['UnitPrice'] = pd.to_numeric(df_od['UnitPrice'], errors='coerce')
    df_od['Quantity'] = pd.to_numeric(df_od['Quantity'], errors='coerce')
    df_od['Discount'] = pd.to_numeric(df_od['Discount'], errors='coerce')
    
    # پر کردن مقادیر خالی تخفیف با صفر تا ارزش سفارش از بین نرود
    # همچنین قیمت و تعداد خالی را نیز می‌توان با صفر پر کرد تا خطای محاسباتی رخ ندهد
    unit_price = df_od['UnitPrice'].fillna(0)
    quantity = df_od['Quantity'].fillna(0)
    discount = df_od['Discount'].fillna(0)
    
    # محاسبه ویژگی ارزش سفارش و گرد کردن به دو رقم اعشار برای استانداردهای مالی
    df_od['OrderValue'] = (unit_price * quantity * (1 - discount)).round(2)
    
    datasets['Order_Details'] = df_od

print("--- OrderValue feature creation complete. ---")


In [ ]:
print("--- 4. Creating Customer_Features Table ---")

if 'Orders' in datasets and 'Order_Details' in datasets:
    # گرفتن کپی از دیتافریم‌ها برای جلوگیری از تغییر ناخواسته در جداول اصلی
    df_orders = datasets['Orders'].copy()
    df_od = datasets['Order_Details'].copy()
    
    # ادغام جدول سفارشات و جزئیات سفارشات
    df_merged = pd.merge(df_orders, df_od, on='OrderID', how='inner')
    
    # محاسبه ارزش ریالی/دلاری تخفیف برای هر سطر
    df_merged['DiscountValue'] = df_merged['UnitPrice'] * df_merged['Quantity'] * df_merged['Discount']
    
    # اطمینان از فرمت تاریخ برای محاسبه روزهای گذشته
    df_merged['OrderDate'] = pd.to_datetime(df_merged['OrderDate'])
    
    # گروه‌بندی بر اساس مشتری و محاسبه ویژگی‌های تجمعی
    customer_features = df_merged.groupby('CustomerID').agg(
        TotalOrders=('OrderID', 'nunique'),           # تعداد سفارش یکتا
        TotalSpending=('OrderValue', 'sum'),          # مجموع هزینه کرد
        TotalItemsPurchased=('Quantity', 'sum'),      # مجموع تعداد کالاها
        UniqueProductsBought=('ProductID', 'nunique'),# تعداد محصولات یکتا
        TotalDiscountReceived=('DiscountValue', 'sum'),# مجموع مبلغ تخفیف گرفته شده
        LastOrderDate=('OrderDate', 'max')            # تاریخ آخرین سفارش (برای محاسبه ریکانسی)
    ).reset_index()
    
    # محاسبه میانگین ارزش هر سفارش
    customer_features['AvgOrderValue'] = customer_features['TotalSpending'] / customer_features['TotalOrders']
    
    # محاسبه تعداد روزهای گذشته از آخرین خرید (با فرض اینکه تاریخ امروز، آخرین تاریخ موجود در کل دیتاست است)
    max_dataset_date = df_orders['OrderDate'].max()
    customer_features['DaysSinceLastOrder'] = (max_dataset_date - customer_features['LastOrderDate']).dt.days
    
    # حذف ستون تاریخ آخرین سفارش چون دیگر نیازی به آن نداریم
    customer_features = customer_features.drop(columns=['LastOrderDate'])
    # گرد کردن مقادیر مالی به دو رقم اعشار
    financial_cols = ['TotalSpending', 'TotalDiscountReceived', 'AvgOrderValue']
    customer_features[financial_cols] = customer_features[financial_cols].round(2)
    
    # اضافه کردن جدول جدید به دیکشنری
    datasets['Customer_Features'] = customer_features
    
    print(f"Customer_Features table created successfully with {customer_features.shape[0]} customers and {customer_features.shape[1]} features.")

else:
    print("Error: 'Orders' or 'Order_Details' tables are missing in the datasets dictionary.")

print("--- Customer_Features table creation complete. ---")


In [ ]:
print("--- 5. Creating StockValue and PriceCategory Features ---")

if 'Products' in datasets:
    df_products = datasets['Products']
    
    # اطمینان از عددی بودن ستون‌ها
    df_products['UnitPrice'] = pd.to_numeric(df_products['UnitPrice'], errors='coerce')
    df_products['UnitsInStock'] = pd.to_numeric(df_products['UnitsInStock'], errors='coerce')
    
    # 1. محاسبه ویژگی ارزش انبار
    df_products['StockValue'] = (df_products['UnitPrice'] * df_products['UnitsInStock']).round(2)
    
    # 2. ایجاد دسته قیمتی (Low, Medium, High) بر اساس توزیع قیمت‌ها
    # مقادیر NaN (در صورت وجود) نادیده گرفته می‌شوند
    df_products['PriceCategory'] = pd.qcut(
        df_products['UnitPrice'][df_products['UnitPrice'] > 0], 
        q=3, 
        labels=['Low', 'Medium', 'High']
    )
    
    datasets['Products'] = df_products
    print("StockValue and PriceCategory features created successfully.")

else:
    print("Error: 'Products' table is missing.")

print("--- Feature creation complete. ---")


In [ ]:
print("--- 6. Checking Normalization Impact on Distribution ---")

# خواندن مجدد داده خام برای مقایسه (چون در بخش ۱ مقادیر را تغییر دادیم)
# در اینجا ستون Freight از جدول Orders را به عنوان نمونه بررسی می‌کنیم
df_sample = pd.read_csv(DATA_DIR / "Orders.csv")
df_sample['Freight'] = pd.to_numeric(df_sample['Freight'], errors='coerce')

# محاسبه متغیر استاندارد شده (Z-Score) به صورت دستی برای مقایسه
mean_f = df_sample['Freight'].mean()
std_f = df_sample['Freight'].std()
df_sample['Freight_Scaled'] = (df_sample['Freight'] - mean_f) / std_f

# ایجاد یک قاب با دو نمودار کنار هم
fig, axes = plt.subplots(1, 2, figsize=(12, 5))

# نمودار اول: توزیع مقادیر اصلی
df_sample['Freight'].plot(kind='hist', bins=40, ax=axes[0], color='skyblue', edgecolor='black')
axes[0].set_title('Original Distribution (Freight)')
axes[0].set_xlabel('Value (Original)')

# نمودار دوم: توزیع مقادیر نرمال‌شده
df_sample['Freight_Scaled'].plot(kind='hist', bins=40, ax=axes[1], color='lightgreen', edgecolor='black')
axes[1].set_title('Scaled Distribution (Z-Score)')
axes[1].set_xlabel('Scaled Value')

plt.suptitle('Comparison: Original vs Scaled Data Distribution', fontsize=14)
plt.tight_layout()
plt.show()
print("Conclusion: Normalization (Z-score/Min-Max) does NOT change the shape of the distribution.")
print("It only changes the scale (X-axis values).")
print("--- Distribution Check Complete. ---")


<div align="center" style="background-color: #E8F8F5; padding: 15px; border-radius: 12px; border-right: 5px solid #1ABC9C; border-left: 5px solid #1ABC9C;">
  <h2 style="color: #0E6655; margin: 0;">
📊 بخش 4 (3.5): شناسایی و مدیریت داده‌های پرت
  </h2>
</div>

In [ ]:
print("\n### Cleaning Raw Data (Orders, Order_Details, Products) ###")

# ---  Fixing Data (اصلاح داده‌ها) ---
print(" Fixing typological errors (Negatives & Zeros)...")

# اصلاح مقادیر منفی در جدول جزئیات سفارش (با قدر مطلق)
datasets['Order_Details']['UnitPrice'] = datasets['Order_Details']['UnitPrice'].abs()
datasets['Order_Details']['Quantity'] = datasets['Order_Details']['Quantity'].abs()

# جایگزینی تعداد سفارش صفر با یک (چون ثبت سفارش به معنای حداقل یک کالاست)
datasets['Order_Details']['Quantity'] = datasets['Order_Details']['Quantity'].replace(0, 1)

# اصلاح هزینه حمل در جدول سفارشات
datasets['Orders']['Freight'] = datasets['Orders']['Freight'].abs()

# اصلاح موجودی منفی انبار به صفر
datasets['Products'].loc[datasets['Products']['UnitsInStock'] < 0, 'UnitsInStock'] = 0

# --- 1.2 Nullifying Data (تبدیل به تهی به جای حذف سطر) ---
print(" Handling logical conflicts (Dates & Missing values)...")

# تاریخ ارسال نمی‌تواند قبل از تاریخ سفارش باشد
invalid_dates_mask = datasets['Orders']['ShippedDate'] < datasets['Orders']['OrderDate']
datasets['Orders'].loc[invalid_dates_mask, 'ShippedDate'] = pd.NaT

print("Raw data cleaning completed successfully.")


In [ ]:
print("\n### Cleaning Constructed Columns & Tables ###")

# ---  Cleaning Custom Columns in Raw Tables ---
print(" Checking custom columns (OrderValue, StockValue)...")

if 'OrderValue' in datasets['Order_Details'].columns:
    # ارزش سفارش نباید منفی باشد
    invalid_order_value = datasets['Order_Details']['OrderValue'] < 0
    datasets['Order_Details'].loc[invalid_order_value, 'OrderValue'] = pd.NA

if 'StockValue' in datasets['Products'].columns:
    # ارزش انبار نباید منفی باشد
    invalid_stock_value = datasets['Products']['StockValue'] < 0
    datasets['Products'].loc[invalid_stock_value, 'StockValue'] = pd.NA

# ---  Cleaning the Custom Table (Customer_Features) ---
print(" Checking Customer_Features table...")

# فرض می‌کنیم جدول مشتریان را با نام Customer_Features ذخیره کرده‌اید
# اگر نام دیگری دارد، در خطوط زیر تغییر دهید
cf_key = 'Customer_Features' if 'Customer_Features' in datasets else 'customer_features'

if cf_key in datasets:
    # الف) ستون‌های مالی: هیچ‌کدام نباید منفی باشند (اگر منفی بودند، صفر در نظر گرفته شوند)
    financial_cols = ['TotalSpending', 'AvgOrderValue', 'TotalDiscountReceived']
    for col in financial_cols:
        if col in datasets[cf_key].columns:
            datasets[cf_key].loc[datasets[cf_key][col] < 0, col] = 0
            
    # ب) ستون‌های شمارشی: تعداد سفارش نباید صفر یا منفی باشد
    count_cols = ['TotalOrders', 'TotalItemsPurchased']
    for col in count_cols:
        if col in datasets[cf_key].columns:
            # اگر صفر یا منفی بود، به عنوان دیتای نامعتبر (NA) مارک شود
            datasets[cf_key].loc[datasets[cf_key][col] <= 0, col] = pd.NA

    # ج) ویژگی‌های زمانی: روزهای گذشته نمی‌تواند منفی باشد
    if 'DaysSinceLastOrder' in datasets[cf_key].columns:
        invalid_days = datasets[cf_key]['DaysSinceLastOrder'] < 0
        datasets[cf_key].loc[invalid_days, 'DaysSinceLastOrder'] = pd.NA

    print("Constructed features cleaning completed successfully.")
else:
    print("Table for Customer Features not found in datasets dictionary.")


In [ ]:
# 3.5: مهندسی و مدیریت داده‌های پرت

print("\n### 1. Outlier Detection (Visualization Only) ###")

# لیست ۵ ستون هدف و جداول مربوطه برای بررسی داده‌های پرت
target_columns = [
    ('Order_Details', 'UnitPrice'),
    ('Order_Details', 'Quantity'),
    ('Orders', 'Freight'),
    ('Order_Details', 'OrderValue'),
    ('Products', 'StockValue')
]

# ایجاد صفحه برای رسم ۵ نمودار جعبه‌ای به صورت افقی (عمودی چیده‌شده)
fig, axes = plt.subplots(5, 1, figsize=(14, 20))
fig.suptitle('Boxplots for Outlier Observation', fontsize=16, fontweight='bold', y=0.995)

# حلقه برای پیمایش ستون‌ها و رسم نمودار جعبه‌ای برای هر کدام
for idx, (table, col) in enumerate(target_columns):
    if table in datasets and col in datasets[table].columns:
        # استخراج داده‌ها و حذف مقادیر خالی برای جلوگیری از خطا در رسم نمودار
        data = datasets[table][col].dropna()
        
        # رسم نمودار جعبه‌ای افقی با استایل بهتر
        bp = axes[idx].boxplot(data, vert=False, patch_artist=True, 
                               boxprops=dict(facecolor='lightblue', color='darkblue', linewidth=1.5),
                               whiskerprops=dict(color='darkblue', linewidth=1.5),
                               capprops=dict(color='darkblue', linewidth=1.5),
                               medianprops=dict(color='red', linewidth=2),
                               flierprops=dict(marker='o', markerfacecolor='orange', markersize=6, 
                                             markeredgecolor='darkorange', alpha=0.7))
        
        # تنظیم عنوان و برچسب‌ها
        axes[idx].set_title(f'{col} ({table})', fontsize=12, fontweight='bold', pad=10, loc='left')
        axes[idx].set_xlabel('Value', fontsize=10)
        axes[idx].grid(axis='x', alpha=0.3, linestyle='--')
        axes[idx].set_yticks([])
        
        print(f"Boxplot successfully generated for '{col}' in '{table}'.")
    else:
        print(f"Warning: Column '{col}' not found in table '{table}'.")

# مرتب‌سازی فاصله‌ها و نمایش نمودارها
plt.tight_layout()
plt.show()

print("\nOutliers visualized using horizontal boxplots.")


In [ ]:
print("\n--- 2. Outliers Count (IQR Method) ---")

# تعریف لیست ستون‌ها و دیتافریم‌های مربوطه برای بررسی داده‌های پرت
columns_to_check = [
    ('Order_Details', 'UnitPrice'),
    ('Order_Details', 'Quantity'),
    ('Orders', 'Freight'),
    ('Order_Details', 'OrderValue'),
    ('Products', 'StockValue')
]

for df_name, col_name in columns_to_check:
    if col_name in datasets[df_name].columns:
        # استخراج داده‌های ستون به صورت عددی و حذف مقادیر خالی
        data = pd.to_numeric(datasets[df_name][col_name], errors='coerce').dropna()
        
        # محاسبه چارک اول و سوم
        Q1 = data.quantile(0.25)
        Q3 = data.quantile(0.75)
        
        # محاسبه دامنه بین چارکی
        IQR = Q3 - Q1
        
        # تعریف محدوده‌های پایین و بالا
        lower_bound = Q1 - 1.5 * IQR
        upper_bound = Q3 + 1.5 * IQR
        
        # شمارش داده‌های پرت
        outliers_count = data[(data < lower_bound) | (data > upper_bound)].count()
        
        print(f"Number of outliers in '{col_name}' ({df_name}): {outliers_count}")


In [ ]:
print("\n--- 3. Outliers Count (Z-Score Method) ---")

# همان لیست ستون‌های قبلی
columns_to_check = [
    ('Order_Details', 'UnitPrice'),
    ('Order_Details', 'Quantity'),
    ('Orders', 'Freight'),
    ('Order_Details', 'OrderValue'),
    ('Products', 'StockValue')
]

for df_name, col_name in columns_to_check:
    if col_name in datasets[df_name].columns:
        # استخراج داده‌های ستون به صورت عددی و حذف مقادیر خالی
        data = pd.to_numeric(datasets[df_name][col_name], errors='coerce').dropna()
        
        # محاسبه امتیاز استاندارد (Z-Score)
        z_scores = (data - data.mean()) / data.std()
        
        # شمارش داده‌های پرت (مقادیری که قدر مطلق Z-Score آنها بیشتر از 3 است)
        outliers_count = data[np.abs(z_scores) > 3].count()
        
        print(f"Number of outliers in '{col_name}' ({df_name}) using Z-Score: {outliers_count}")


In [ ]:
import numpy as np
import pandas as pd

print("\n--- 4. Outliers Comparison (IQR vs Z-Score) ---")

# تعریف لیست ستون‌ها و دیتافریم‌های مربوطه
columns_to_check = [
    ('Order_Details', 'UnitPrice'),
    ('Order_Details', 'Quantity'),
    ('Orders', 'Freight'),
    ('Order_Details', 'OrderValue'),
    ('Products', 'StockValue')
]

# لیستی برای ذخیره نتایج جهت ساخت جدول مقایسه‌ای
comparison_results = []

for df_name, col_name in columns_to_check:
    if df_name in datasets and col_name in datasets[df_name].columns:
        # استخراج داده‌ها و پاک‌سازی اولیه
        data = pd.to_numeric(datasets[df_name][col_name], errors='coerce').dropna()
        
        # --- محاسبه با روش IQR ---
        Q1 = data.quantile(0.25)
        Q3 = data.quantile(0.75)
        IQR = Q3 - Q1
        lower_bound = Q1 - 1.5 * IQR
        upper_bound = Q3 + 1.5 * IQR
        iqr_outliers = data[(data < lower_bound) | (data > upper_bound)].count()
        
        # --- محاسبه با روش Z-Score ---
        z_scores = (data - data.mean()) / data.std()
        z_score_outliers = data[np.abs(z_scores) > 3].count()
        
        # افزودن به لیست نتایج
        comparison_results.append({
            'Table': df_name,
            'Column': col_name,
            'IQR_Outliers': iqr_outliers,
            'Z-Score_Outliers': z_score_outliers
        })

# تبدیل به دیتافریم برای نمایش زیباتر به شکل جدول
comparison_df = pd.DataFrame(comparison_results)
print(comparison_df.to_string(index=False))

In [ ]:


print("\n--- 5. Handling Outliers using Winsorization (Percentile Capping) ---")

cols_to_winsorize = [
    ('Order_Details', 'UnitPrice'),
    ('Order_Details', 'Quantity'),
    ('Orders', 'Freight'),
    ('Order_Details', 'OrderValue'),
    ('Products', 'StockValue')
]

# 💾 تهیه نسخه پشتیبان از داده‌های اصلی قبل از هرگونه تغییر جهت رسم نمودارهای مقایسه‌ای واقعی
freight_before = datasets['Orders']['Freight'].copy()
quantity_before = datasets['Order_Details']['Quantity'].copy()
price_before = datasets['Order_Details']['UnitPrice'].copy()
for df_name, col_name in cols_to_winsorize:
    if df_name in datasets and col_name in datasets[df_name].columns:
        col_data = pd.to_numeric(datasets[df_name][col_name], errors='coerce')
        
        # تعیین مرزها بر اساس صدک ۵ و ۹۵ برای حفظ تعادل توزیع داده‌ها
        lower_limit = col_data.quantile(0.05)
        upper_limit = col_data.quantile(0.95)
        
        # اعمال تغییرات روی دیتای اصلی
        datasets[df_name][col_name] = col_data.clip(lower=lower_limit, upper=upper_limit)
        
        print(f"* Winsorization applied to '{col_name}' in '{df_name}' (Limits: {lower_limit:.2f} to {upper_limit:.2f})")

print("\n✅ Winsorization completed successfully!")

# ---------------------------------------------------------
# رسم نمودار مقایسه‌ای Freight (قبل و بعد از وینزوریزاسیون)
# ---------------------------------------------------------
print("\n--- Visualizing the effect of Winsorization on 'Orders.Freight' ---")

freight_after = datasets['Orders']['Freight']

fig, axes = plt.subplots(1, 2, figsize=(12, 5))

# نمودار قبل از تغییر (داده‌های واقعی و اصلی)
axes[0].hist(freight_before.dropna(), bins=30, color='skyblue', edgecolor='black')
axes[0].set_title('Before Winsorization (Freight - Original)')
axes[0].set_xlabel('Value')
axes[0].set_ylabel('Frequency')

# نمودار بعد از تغییر
axes[1].hist(freight_after.dropna(), bins=30, color='lightgreen', edgecolor='black')
axes[1].set_title('After Winsorization (Freight - Capped)')
axes[1].set_xlabel('Value')
axes[1].set_ylabel('Frequency')

plt.suptitle('Effect of Winsorization (5th & 95th Percentiles)', fontsize=14)
plt.tight_layout()
plt.show()

In [ ]:
# ---------------------------------------------------------
# ۲۳. بررسی و مدیریت سفارش‌هایی با تعداد بسیار زیاد (Quantity)
# ---------------------------------------------------------

print("\n--- 23. Checking & Handling Outliers in Order Quantities ---")

# استفاده از داده‌های اولیه (پشتیبان‌گیری شده در بخش دوم) برای محاسبات و نمودار.
# این کار مانع از تأثیرپذیری محاسبات از وینزوریزاسیون کلی بخش دوم می‌شود.
Q1_q = quantity_before.quantile(0.25)
Q3_q = quantity_before.quantile(0.75)
IQR_q = Q3_q - Q1_q
upper_bound_q = Q3_q + 1.5 * IQR_q

# شناسایی ردیف‌های پرت واقعی بر اساس داده‌های اولیه
high_quantity_orders = datasets['Order_Details'][quantity_before > upper_bound_q]

if not high_quantity_orders.empty:
    msg_detected = f"⚠️ Detected {len(high_quantity_orders)} outlier rows with exceptionally high quantities."
    msg_bound = f"Upper bound for normal data based on IQR: {upper_bound_q}"
    msg_action = "✅ Management Decision: Quantity outliers successfully clipped to the IQR upper bound."
    
    if 'log' in globals():
        log(msg_detected)
        print(msg_bound)
        log(msg_action)
    else:
        print(msg_detected)
        print(msg_bound)
        print(msg_action)
else:
    print("✅ No abnormal orders with exceptionally high quantities found.")

# ---------------------------------------------------------
# رسم نمودار جعبه‌ای برای داده‌های اولیه Quantity
# ---------------------------------------------------------
plt.figure(figsize=(8, 4))
plt.boxplot(
    quantity_before.dropna(),  # رسم بر روی داده‌های اولیه برای نمایش درست نقاط پرت (Red Dots)
    vert=False,
    patch_artist=True,
    boxprops=dict(facecolor='lightcoral', color='black'),
    flierprops=dict(marker='o', markerfacecolor='red', markersize=6, linestyle='none')
)
plt.title('Outliers Detection in Quantities (Original Data)', fontsize=12)
plt.xlabel('Quantity')
plt.grid(axis='x', linestyle='--', alpha=0.7)
plt.tight_layout()
plt.show()

In [ ]:
# ---------------------------------------------------------
# ۲۴. بررسی و مدیریت محصولات با قیمت غیرعادی بالا یا پایین (Products.UnitPrice)
# ---------------------------------------------------------
print("\n--- 24. Checking & Handling Outliers in Product Unit Prices ---")

# مطمئن می‌شویم که داده‌ها عددی هستند و مقادیر خالی حذف شده‌اند
raw_prices = pd.to_numeric(datasets['Products']['UnitPrice'], errors='coerce')

# محاسبه میانگین و انحراف معیار داده‌های فعلی برای سنجش Z-Score
mean_price = raw_prices.mean()
std_price = raw_prices.std()
z_scores = (raw_prices - mean_price) / std_price

print(f"Original Price Mean: {mean_price:.4f}")
print(f"Original Price Std Dev: {std_price:.4f}")

# تعیین مرز علمی استاندارد (Z-Score > 2.5 برای داده‌های دارای چولگی مناسب است)
z_threshold = 2.5
print(f"Z-Score Threshold set to: +/- {z_threshold}")

# شناسایی داده‌های پرت واقعی بر اساس آستانه Z-Score
low_price_outliers = datasets['Products'][z_scores < -z_threshold]
high_price_outliers = datasets['Products'][z_scores > z_threshold]

# الف) مدیریت قیمت‌های غیرعادی پایین
if not low_price_outliers.empty:
    msg_low = f"⚠️ Detected {len(low_price_outliers)} products with abnormally low prices (Z < -{z_threshold})."
    if 'log' in globals(): log(msg_low)
    else: print(msg_low)
    
    print("Products with abnormally low prices before correction:")
    print(low_price_outliers[['ProductID', 'ProductName', 'UnitPrice']])
    
    # جایگزینی قیمت‌های نامعتبر پایین با میانه داده‌ها
    median_price = raw_prices.median()
    datasets['Products'].loc[z_scores < -z_threshold, 'UnitPrice'] = median_price
    
    msg_low_action = f"✅ Management Decision: Invalid low prices replaced with data median ({median_price:.4f})."
    if 'log' in globals(): log(msg_low_action)
    else: print(msg_low_action)
else:
    print(f"✅ No products found with abnormally low prices (Z < -{z_threshold}).")

# ب) مدیریت قیمت‌های غیرعادی بالا (قیمت‌های نجومی)
if not high_price_outliers.empty:
    msg_high = f"⚠️ Detected {len(high_price_outliers)} products with abnormally high prices (Z > {z_threshold})."
    if 'log' in globals(): log(msg_high)
    else: print(msg_high)
    
    print("\nTop most expensive products before correction:")
    print(high_price_outliers[['ProductID', 'ProductName', 'UnitPrice']].sort_values(by='UnitPrice', ascending=False))
    
    # محاسبه ارزش عددی مرز بالایی که معادل Z = 2.5 است (در اینجا ۳۰۶.۰۳)
    upper_value_boundary = mean_price + (z_threshold * std_price)
    
    # محدود کردن (Clip) مقادیر پرت به سقف تعیین شده
    datasets['Products']['UnitPrice'] = raw_prices.clip(upper=upper_value_boundary)
    
    msg_high_action = f"✅ Management Decision: High outliers clipped to the value threshold of {upper_value_boundary:.2f} (Z = {z_threshold})."
    if 'log' in globals(): log(msg_high_action)
    else: print(msg_high_action)
else:
    print(f"✅ No products found with abnormally high prices (Z > {z_threshold}).")

In [ ]:
# ---------------------------------------------------------
# بخش X: بررسی تأثیر حذف داده‌های پرت بر میانگین و انحراف معیار
# ---------------------------------------------------------
print("\n--- Impact of Removing Outliers on Mean and Standard Deviation ---")

# تعریف ستون‌ها؛ این بار از متغیرهای بک‌آپ (داده‌های خام اولیه) استفاده می‌کنیم تا تداخل رفع شود
# نکته: مطمئن شوید متغیرهای قیمت و کرایه قبل از وینزوریزاسیون کپی شده باشند (مانند قیمت و تعداد)
columns_to_check = [
    {'dataset': 'Orders', 'col': 'Freight', 'raw_data': freight_before if 'freight_before' in globals() else datasets['Orders']['Freight']},
    {'dataset': 'Order_Details', 'col': 'UnitPrice', 'raw_data': price_before if 'price_before' in globals() else datasets['Order_Details']['UnitPrice']},
    {'dataset': 'Order_Details', 'col': 'Quantity', 'raw_data': quantity_before if 'quantity_before' in globals() else datasets['Order_Details']['Quantity']}
]

# لیستی برای ذخیره نتایج جهت نمایش در یک جدول
impact_results = []

for item in columns_to_check:
    df_name = item['dataset']
    col_name = item['col']
    
    # استخراج داده‌های اولیه و حذف مقادیر خالی برای محاسبات دقیق
    data = pd.to_numeric(item['raw_data'], errors='coerce').dropna()
    
    # ۱. محاسبه میانگین و انحراف معیار قبل از حذف داده‌های پرت
    mean_before = data.mean()
    std_before = data.std()
    
    # ۲. شناسایی محدوده‌های مجاز با روش استاندارد IQR
    Q1 = data.quantile(0.25)
    Q3 = data.quantile(0.75)
    IQR = Q3 - Q1
    
    lower_bound = Q1 - 1.5 * IQR
    upper_bound = Q3 + 1.5 * IQR
    
    # ۳. فیلتر کردن و حذف داده‌های پرت خارج از محدوده IQR
    data_without_outliers = data[(data >= lower_bound) & (data <= upper_bound)]
    
    # ۴. محاسبه میانگین و انحراف معیار بعد از حذف داده‌های پرت
    mean_after = data_without_outliers.mean()
    std_after = data_without_outliers.std()
    
    # محاسبه تعداد دقیق داده‌های پرت حذف شده
    outliers_removed_count = len(data) - len(data_without_outliers)
    
    # ذخیره نهایی نتایج در لیست
    impact_results.append({
        'Dataset': df_name,
        'Column': col_name,
        'Outliers Removed': outliers_removed_count,
        'Mean (Before)': round(mean_before, 2),
        'Mean (After)': round(mean_after, 2),
        'Std (Before)': round(std_before, 2),
        'Std (After)': round(std_after, 2)
    })

# تبدیل نتایج به دیتافریم برای نمایش منظم
impact_df = pd.DataFrame(impact_results)

# نمایش جدول نهایی در خروجی
print(impact_df.to_string(index=False))

<div align="center" style="background-color: #E8F8F5; padding: 15px; border-radius: 12px; border-right: 5px solid #1ABC9C; border-left: 5px solid #1ABC9C;">
  <h2 style="color: #0E6655; margin: 0;">
✅ بخش 6 (3.6): اعتبارسنجی داده‌ها
  </h2>
</div>

In [ ]:
# ==========================================
# Section 13: Data Validation (اعتبارسنجی داده‌ها)
# 13.1 Logical Sequence of Dates
# ==========================================
print("\n--- 6.3 Data Validation: Logical Sequence of Dates ---")

# تعریف دیکشنری گزارش کیفیت داده‌ها در صورتی که از قبل تعریف نشده باشد
if 'validation_summary' not in globals():
    validation_summary = {}

# استخراج جدول سفارشات
orders_df = datasets['Orders']

# اطمینان از اینکه فرمت ستون‌ها حتماً DateTime باشد تا مقایسه درست انجام شود
orders_df['OrderDate'] = pd.to_datetime(orders_df['OrderDate'], errors='coerce')
orders_df['ShippedDate'] = pd.to_datetime(orders_df['ShippedDate'], errors='coerce')
orders_df['RequiredDate'] = pd.to_datetime(orders_df['RequiredDate'], errors='coerce')

# ایجاد ماسک‌های منطقی برای پیدا کردن خطاهای تاریخ
mask_invalid_shipped = orders_df['ShippedDate'] < orders_df['OrderDate']
mask_invalid_required = orders_df['RequiredDate'] < orders_df['OrderDate']

# شمارش خطاها
count_invalid_shipped = mask_invalid_shipped.sum()
count_invalid_required = mask_invalid_required.sum()

# بررسی و اصلاح تاریخ‌های ارسال (ShippedDate)
print(f"Orders where ShippedDate is before OrderDate: {count_invalid_shipped}")
if count_invalid_shipped > 0:
    affected_orders = orders_df.loc[mask_invalid_shipped, 'OrderID'].tolist()
    print(f"   -> Sample of affected OrderIDs: {affected_orders[:5]}")
    
    # 🛠️ اعمال جایگزینی واقعی با مقدار خالی (pd.NaT) روی دیتاست اصلی
    datasets['Orders'].loc[mask_invalid_shipped, 'ShippedDate'] = pd.NaT
    print("   -> [Fixed]: Invalid ShippedDates have been replaced with pd.NaT.")
else:
    print("   -> No logical anomalies found in ShippedDates.")

# بررسی و اصلاح تاریخ‌های موردنیاز (RequiredDate)
print(f"\nOrders where RequiredDate is before OrderDate: {count_invalid_required}")
if count_invalid_required > 0:
    affected_orders = orders_df.loc[mask_invalid_required, 'OrderID'].tolist()
    print(f"   -> Sample of affected OrderIDs: {affected_orders[:5]}")
    
    # جایگزینی تاریخ‌های نامعتبر با مقدار خالی (pd.NaT)
    datasets['Orders'].loc[mask_invalid_required, 'RequiredDate'] = pd.NaT
    print("   -> [Fixed]: Invalid RequiredDates have been replaced with pd.NaT.")
else:
    print("   -> No logical anomalies found in RequiredDates.")

# ذخیره نتایج در خلاصه وضعیت برای گزارش کیفیت داده نهایی
validation_summary['Invalid ShippedDate'] = {
    'Errors Found': count_invalid_shipped, 
    'Corrections Applied': count_invalid_shipped
}
validation_summary['Invalid RequiredDate'] = {
    'Errors Found': count_invalid_required, 
    'Corrections Applied': count_invalid_required
}

print("\n✅ Date sequence validation completed successfully.")

In [ ]:
# ==========================================
# Section 13: Data Validation (اعتبارسنجی داده‌ها)
# 13.2 Data Validation: Referential Integrity (Order IDs)
# ==========================================
print("\n--- 6.3 Data Validation: Referential Integrity (Order IDs) ---")

# استخراج جداول سفارشات و جزئیات سفارشات
orders_df = datasets['Orders']
order_details_df = datasets['Order_Details']

# استخراج مجموعه (Set) شناسه‌های یکتا برای مقایسه سریع‌تر و بهینه‌تر
valid_order_ids = set(orders_df['OrderID'])
details_order_ids = set(order_details_df['OrderID'])

# 🛠️ [رفع باگ]: خط اضافی و اشتباه مربوط به ShippedDate از این قسمت حذف شد.

# پیدا کردن شناسه‌هایی که در جزئیات هستند اما در جدول اصلی سفارشات وجود ندارند (رکوردهای یتیم)
orphan_order_ids = details_order_ids - valid_order_ids

print(f"Total unique OrderIDs in Orders table: {len(valid_order_ids)}")
print(f"Invalid unique OrderIDs in Order Details table (Orphan IDs): {len(orphan_order_ids)}")

orphan_rows_count = 0

if len(orphan_order_ids) > 0:
    print(f"   -> Sample of invalid OrderIDs: {list(orphan_order_ids)[:5]}")
    
    # پیدا کردن ماسک و تعداد کل سطرهای یتیم در جدول جزئیات سفارش
    orphan_mask = order_details_df['OrderID'].isin(orphan_order_ids)
    orphan_rows_count = orphan_mask.sum()
    print(f"   -> Total orphan rows in Order Details table: {orphan_rows_count}")
    
    # حذف سطرهای یتیم از جدول جزئیات سفارش برای حفظ یکپارچگی مرجع (Referential Integrity)
    datasets['Order_Details'] = order_details_df[~orphan_mask].reset_index(drop=True)
    print("   -> [Fixed]: Orphan rows with invalid OrderIDs have been removed from Order Details.")
else:
    print("   -> [Verified]: Referential integrity is intact. All OrderIDs in Order Details exist in Orders table.")

# ذخیره نتایج در دیکشنری گزارش کیفیت داده‌ها
validation_summary['Referential Integrity (OrderID)'] = {
    'Errors Found': orphan_rows_count,
    'Corrections Applied': orphan_rows_count
}

In [ ]:
# ==========================================
# Section 13: Data Validation (اعتبارسنجی داده‌ها)
# 13.3 Data Validation: Logic Checks & Ranges
# ==========================================
print("\n" + "="*65)
print(" [⚙️] Executing Validation: Logical Value & Range Checks")
print("="*65)

# تعریف دیکشنری گزارش کیفیت داده‌ها در صورتی که از قبل تعریف نشده باشد
if 'validation_summary' not in globals():
    validation_summary = {}

# ۱. ستون‌هایی که باید اکیداً بزرگ‌تر از صفر باشند
strictly_positive_rules = {
    'Order_Details': ['Quantity']
}

# ۲. ستون‌هایی که نباید منفی باشند (بزرگتر یا مساوی صفر)
non_negative_rules = {
    'Order_Details': ['UnitPrice'],
    'Orders': ['Freight'],
    'Products': ['UnitPrice', 'UnitsInStock', 'UnitsOnOrder', 'ReorderLevel']
}

# ۳. ستون‌هایی که باید دقیقاً در بازه درصد [0, 1] باشند
percentage_rules = {
    'Order_Details': ['Discount']
}

# --- بخش اول: بررسی مقادیر اکیداً مثبت ---
for table, columns in strictly_positive_rules.items():
    if table in datasets:
        for col in columns:
            if col in datasets[table].columns:
                datasets[table][col] = pd.to_numeric(datasets[table][col], errors='coerce')
                invalid_mask = datasets[table][col] <= 0
                invalid_count = invalid_mask.sum()
                
                # ثبت در گزارش کیفیت داده‌ها
                validation_summary[f"{table}.{col} (<=0)"] = {
                    'Errors Found': invalid_count,
                    'Corrections Applied': invalid_count
                }
                
                if invalid_count > 0:
                    print(f"  [!] Warning in table '{table}' | column '{col}':")
                    print(f"      >> Found {invalid_count} records with invalid values (<= 0).")
                    
                    # 🛠️ اصلاح مستقیم روی دیتابیس اصلی: منفی‌ها به قدر مطلق و صفرها به ۱
                    datasets[table].loc[datasets[table][col] < 0, col] = datasets[table].loc[datasets[table][col] < 0, col].abs()
                    datasets[table].loc[datasets[table][col] == 0, col] = 1
                    print(f"      [✔] Action applied: Negative values converted to Absolute, zeros set to 1.")
                else:
                    print(f"  [✓] Table '{table}' | column '{col}': All values are strictly positive.")

# --- بخش دوم: بررسی مقادیر نامنفی (>= 0) ---
for table, columns in non_negative_rules.items():
    if table in datasets:
        for col in columns:
            if col in datasets[table].columns:
                datasets[table][col] = pd.to_numeric(datasets[table][col], errors='coerce')
                invalid_mask = datasets[table][col] < 0
                invalid_count = invalid_mask.sum()
                
                # ثبت در گزارش کیفیت داده‌ها
                validation_summary[f"{table}.{col} (<0)"] = {
                    'Errors Found': invalid_count,
                    'Corrections Applied': invalid_count
                }
                
                if invalid_count > 0:
                    print(f"  [!] Warning in table '{table}' | column '{col}':")
                    print(f"      >> Found {invalid_count} records with negative values.")
                    
                    # 🛠️ اصلاح بر اساس نوع ستون (ستون‌های موجودی انبار صفر می‌شوند، قیمت‌ها قدر مطلق)
                    if col in ['UnitsInStock', 'UnitsOnOrder', 'ReorderLevel']:
                        datasets[table].loc[invalid_mask, col] = 0
                        print(f"      [✔] Action applied: Stock/Inventory negatives set to 0.")
                    else:
                        datasets[table].loc[invalid_mask, col] = datasets[table].loc[invalid_mask, col].abs()
                        print(f"      [✔] Action applied: Financial negatives converted to Absolute values.")
                else:
                    print(f"  [✓] Table '{table}' | column '{col}': All values are non-negative.")

# --- بخش سوم: بررسی بازه تخفیف (بین ۰ و ۱) ---
for table, columns in percentage_rules.items():
    if table in datasets:
        for col in columns:
            if col in datasets[table].columns:
                datasets[table][col] = pd.to_numeric(datasets[table][col], errors='coerce')
                invalid_mask = (datasets[table][col] < 0) | (datasets[table][col] > 1)
                invalid_count = invalid_mask.sum()
                
                # ثبت در گزارش کیفیت داده‌ها
                validation_summary[f"{table}.{col} (out of [0,1])"] = {
                    'Errors Found': invalid_count,
                    'Corrections Applied': invalid_count
                }
                
                if invalid_count > 0:
                    print(f"  [!] Warning in table '{table}' | column '{col}':")
                    print(f"      >> Found {invalid_count} records outside the logical [0, 1] range.")
                    
                    # 🛠️ اصلاح محدوده با استفاده از تابع clip روی منبع اصلی
                    datasets[table][col] = datasets[table][col].clip(lower=0.0, upper=1.0)
                    print(f"      [✔] Action applied: Values capped/clipped within [0.0, 1.0] range.")
                else:
                    print(f"  [✓] Table '{table}' | column '{col}': All values are within valid percentage limits.")

print("-" * 65)
print(" [✅] Logical range and value validation completed successfully.")
print("=" * 65 + "\n")

In [ ]:
# ==========================================
# Section 13: Data Validation (اعتبارسنجی داده‌ها)
# 13.4 Data Validation: Referential Integrity (Products -> Categories)
# ==========================================
print("\n" + "="*65)
print(" [🔗] Executing Validation: Referential Integrity (Products -> Categories)")
print("="*65)

# تعریف دیکشنری گزارش کیفیت داده‌ها در صورتی که از قبل تعریف نشده باشد
if 'validation_summary' not in globals():
    validation_summary = {}

if 'Products' in datasets and 'Categories' in datasets:
    products_df = datasets['Products']
    categories_df = datasets['Categories']
    
    # استخراج شناسه‌های معتبر دسته‌بندی از جدول Categories
    valid_category_ids = categories_df['CategoryID'].dropna().unique()
    
    # بررسی شناسه‌های نامعتبر در جدول محصولات (با نادیده گرفتن مقادیر خالی پیش‌فرض)
    invalid_mask = products_df['CategoryID'].notna() & ~products_df['CategoryID'].isin(valid_category_ids)
    invalid_count = invalid_mask.sum()
    
    # ثبت نتایج در دیکشنری گزارش کیفیت داده‌ها
    validation_summary['Referential Integrity (CategoryID)'] = {
        'Errors Found': invalid_count,
        'Corrections Applied': invalid_count
    }
    
    if invalid_count > 0:
        print(f"  [!] Warning in table 'Products':")
        print(f"      >> Found {invalid_count} products with invalid 'CategoryID' (missing in Categories table).")
        
        # 🛠️ اقدام اصلاحی مستقیم روی دیتاست اصلی: تبدیل شناسه‌های نامعتبر به NaN
        datasets['Products'].loc[invalid_mask, 'CategoryID'] = np.nan
        print("      [✔] Action applied: Invalid CategoryIDs in Products table have been set to NaN.")
    else:
        print("  [✓] Table 'Products': All products have a valid and existing CategoryID.")
else:
    print("  [!] Error: Products or Categories tables not found in the dataset.")

print("-" * 65 + "\n")

In [ ]:
# =========================================================
# ADVANCED DATA QUALITY REPORT
# =========================================================

def calculate_dataset_quality(df):
    total_cells = df.shape[0] * df.shape[1]

    # Missing values
    missing_count = df.isna().sum().sum()

    # Duplicate rows
    duplicate_rows = df.duplicated().sum()

    # Empty strings in object columns
    empty_strings = 0
    for col in df.select_dtypes(include="object").columns:
        empty_strings += (df[col].astype(str).str.strip() == "").sum()

    issue_count = missing_count + duplicate_rows + empty_strings

    if total_cells == 0:
        quality_percent = 100
    else:
        quality_percent = max(0, (1 - (issue_count / total_cells)) * 100)

    quality_score = round(quality_percent, 2)

    return {
        "rows": df.shape[0],
        "columns": df.shape[1],
        "missing": int(missing_count),
        "duplicates": int(duplicate_rows),
        "empty_strings": int(empty_strings),
        "quality_percent": quality_percent,
        "score": quality_score
    }


def print_data_quality_report(datasets):

    print("\n")
    print("="*75)
    print(" " * 22 + "DATA QUALITY ASSESSMENT REPORT")
    print("="*75)

    global_scores = []

    for name, df in datasets.items():

        stats = calculate_dataset_quality(df)
        global_scores.append(stats["score"])

        print("\n" + "-"*75)
        print(f"Dataset: {name}")
        print("-"*75)

        print(f"Rows                     : {stats['rows']}")
        print(f"Columns                  : {stats['columns']}")
        print(f"Missing Values           : {stats['missing']}")
        print(f"Duplicate Rows           : {stats['duplicates']}")
        print(f"Empty String Values      : {stats['empty_strings']}")

        print("-"*40)

        print(f"Data Quality Percentage  : {stats['quality_percent']:.2f}%")
        print(f"Data Quality Score       : {stats['score']}/100")

        if stats["score"] >= 95:
            status = "Excellent"
        elif stats["score"] >= 85:
            status = "Good"
        elif stats["score"] >= 70:
            status = "Moderate"
        else:
            status = "Poor"

        print(f"Quality Status           : {status}")

    # Global summary
    if global_scores:
        overall_score = sum(global_scores) / len(global_scores)
    else:
        overall_score = 100

    print("\n" + "="*75)
    print("OVERALL DATA QUALITY SUMMARY")
    print("="*75)

    print(f"Datasets Evaluated       : {len(datasets)}")
    print(f"Overall Quality Score    : {overall_score:.2f}/100")

    if overall_score >= 95:
        overall_status = "Excellent"
    elif overall_score >= 85:
        overall_status = "Good"
    elif overall_score >= 70:
        overall_status = "Moderate"
    else:
        overall_status = "Poor"

    print(f"Overall Data Status      : {overall_status}")
    print("="*75 + "\n")


# Run final data quality report
print_data_quality_report(datasets)


# =========================================================
# گزارش خطاها و اصلاحات بخش اعتبارسنجی (VALIDATION ERRORS & CORRECTIONS REPORT)
# =========================================================
print("\n" + "="*75)
print(" " * 15 + "DATA VALIDATION ERRORS & CORRECTIONS REPORT")
print("="*75)

# بررسی اینکه آیا دیکشنری خلاصه خطاها وجود دارد و پر است
if 'validation_summary' in globals() and validation_summary:
    # تبدیل دیکشنری به دیتافریم برای نمایش جدولی و زیبا
    validation_df = pd.DataFrame.from_dict(validation_summary, orient='index')
    
    # بازنشانی اندیس برای اینکه نام خطاها به عنوان یک ستون قرار گیرد
    validation_df.index.name = 'Validation Check Rule'
    validation_df = validation_df.reset_index()
    
    # چاپ جدول به صورت کاملاً انگلیسی و مرتب
    print(validation_df.to_string(index=False))
else:
    print("No validation rules were executed or no summary data is available.")

print("="*75 + "\n")